# 05 — Test-Year Target Grid Alignment (FINAL FIXED v2)

This clean version is built for the current Khulna repository structure.

Key protections:
- Correct year/month filename parsing.
- Actual precipitation folder mapping.
- Only 2022 is aligned for spatial prediction.
- Training/validation rasters are left native.
- No edge clamping.
- No nearest filling of coverage gaps.
- Unusable CDR EngineeringCRS is only treated as WGS84 when raster bounds are clearly geographic.
- QC reports valid coverage before the modelling stage.

`0.005°` is an output grid spacing, not a claim that all inputs contain independent 500 m information.


In [1]:

# ============================================================
# 05 — TEST YEAR TARGET GRID ALIGNMENT (FINAL FIXED v2)
# Khulna Precipitation Downscaling
# ============================================================

from pathlib import Path
import re
import math
import warnings

import numpy as np
import pandas as pd
import rasterio
import geopandas as gpd

from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from rasterio.features import geometry_mask


# ============================================================
# 1. FIND PROJECT ROOT
# ============================================================

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)


# ============================================================
# 2. SETTINGS
# ============================================================

TEST_YEAR = 2022
TARGET_RES_DEG = 0.005
DST_NODATA = -9999.0

# Avoid EPSG database lookup in the user's current environment.
WGS84_PROJ = "+proj=longlat +datum=WGS84 +no_defs"


# ============================================================
# 3. ACTUAL FOLDER MAPPING
# ============================================================

PRECIP_FOLDERS = {
    "CCS": "CCS",
    "PDIR": "PDIR",
    "GSMaP_MVK": "GSMaP_MVK",
    "CDR": "CDR",
    "CHIRPS": "CHIRPS_TIFF_2017_2022",
    "IMERG": "IMERG_Monthly",
    "GSMaP_Gauge": "GSMaP_Gauge_v7",
    "ERA5": "ERA5_TIFF",
}

LAND_DYNAMIC = ["NDVI", "LST_Day"]


# ============================================================
# 4. HELPERS
# ============================================================

def parse_ym(name):
    """
    Parse year/month from filenames such as:
    product_2017_01.tif
    product-2017-01.tif
    product_201701.tif
    """
    stem = Path(name).stem

    patterns = [
        r"(?<!\d)(20\d{2})[_-](0?[1-9]|1[0-2])(?!\d)",
        r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(?!\d)",
    ]

    for pat in patterns:
        m = re.search(pat, stem)
        if m:
            return int(m.group(1)), int(m.group(2))

    return None


def list_rasters(folder):
    if not folder.exists():
        return []
    return sorted([*folder.rglob("*.tif"), *folder.rglob("*.tiff")])


def monthly_map(folder):
    out = {}

    for p in list_rasters(folder):
        ym = parse_ym(p.name)

        if ym is None:
            continue

        if ym in out:
            raise ValueError(
                f"Duplicate raster for {folder.name} {ym}:\n"
                f"{out[ym]}\n{p}"
            )

        out[ym] = p

    return out


def choose_static(folder_name, preferred_names):
    folder = RAW_DIR / "predictors" / folder_name
    files = list_rasters(folder)

    if not files:
        raise FileNotFoundError(f"No raster found for {folder_name}")

    lookup = {p.name.lower(): p for p in files}

    for name in preferred_names:
        if name.lower() in lookup:
            return lookup[name.lower()]

    clean = [
        p for p in files
        if not any(
            k in p.stem.lower()
            for k in ["clip", "tmp", "temp", "aligned", "resampl"]
        )
    ]

    if len(clean) == 1:
        return clean[0]

    if len(files) == 1:
        return files[0]

    raise ValueError(
        f"Ambiguous static predictor {folder_name}:\n"
        + "\n".join(str(p) for p in files)
    )


def source_crs_or_verified_wgs84(src, path):
    """
    Accept a normal geographic/projected CRS.

    If CRS is missing/unusable (e.g., EngineeringCRS) but bounds are clearly
    longitude/latitude, use WGS84. Otherwise stop rather than making a blind guess.
    """
    b = src.bounds

    geographic_bounds = (
        -180 <= b.left <= 180
        and -180 <= b.right <= 180
        and -90 <= b.bottom <= 90
        and -90 <= b.top <= 90
    )

    if src.crs is None:
        if geographic_bounds:
            print(
                f"WARNING: {path.name} has no CRS -> "
                "using WGS84 because bounds are geographic."
            )
            return WGS84_PROJ

        raise ValueError(
            f"{path.name}: CRS missing and bounds are not geographic."
        )

    try:
        is_geo = bool(src.crs.is_geographic)
        is_proj = bool(src.crs.is_projected)
    except Exception:
        is_geo = False
        is_proj = False

    if is_geo or is_proj:
        return src.crs

    if geographic_bounds:
        print(
            f"WARNING: {path.name} has unusable/Engineering CRS -> "
            "using WGS84 because bounds are geographic."
        )
        return WGS84_PROJ

    raise ValueError(
        f"{path.name}: CRS is unusable and bounds are not safely lon/lat."
    )


# ============================================================
# 5. QUICK PARSER SELF-TEST
# ============================================================

_parser_tests = {
    "CCS_2022_01.tif": (2022, 1),
    "abc-2021-12.tif": (2021, 12),
    "IMERG_202203.tif": (2022, 3),
}
for _name, _expected in _parser_tests.items():
    _got = parse_ym(_name)
    if _got != _expected:
        raise RuntimeError(
            f"Year-month parser failed for {_name}: got {_got}, expected {_expected}"
        )

print("Filename year-month parser: OK")


# ============================================================
# 6. CANONICAL STATIC PREDICTORS
# ============================================================

DEM_PATH = choose_static(
    "DEM",
    ["Khulna_SRTM_DEM.tif", "DEM.tif"],
)

DFS_PATH = choose_static(
    "Distance_Sea",
    ["Distance_Sea.tif", "distance_to_sea.tif"],
)

print("\nCanonical DEM:")
print(DEM_PATH)

print("\nCanonical Distance to Sea:")
print(DFS_PATH)


# ============================================================
# 7. STUDY BOUNDARY
# ============================================================

boundary_dir = RAW_DIR / "boundary"

boundary_candidates = [
    *boundary_dir.rglob("*.shp"),
    *boundary_dir.rglob("*.gpkg"),
    *boundary_dir.rglob("*.geojson"),
]

if not boundary_candidates:
    raise FileNotFoundError(
        "No boundary vector found under data/raw/boundary"
    )

boundary_path = boundary_candidates[0]
print("\nBoundary file:")
print(boundary_path)

gdf = gpd.read_file(boundary_path)

if gdf.crs is None:
    raise ValueError("Study boundary has no CRS.")

try:
    gdf = gdf.to_crs(WGS84_PROJ)
except Exception as e:
    raise RuntimeError(
        "Boundary reprojection failed.\n"
        f"Original error:\n{e}"
    )

gdf = gdf[gdf.geometry.notna() & (~gdf.geometry.is_empty)].copy()

if gdf.empty:
    raise ValueError("Boundary contains no valid geometry.")


# ============================================================
# 8. TARGET GRID
# ============================================================

minx, miny, maxx, maxy = gdf.total_bounds

left = math.floor(minx / TARGET_RES_DEG) * TARGET_RES_DEG
right = math.ceil(maxx / TARGET_RES_DEG) * TARGET_RES_DEG
bottom = math.floor(miny / TARGET_RES_DEG) * TARGET_RES_DEG
top = math.ceil(maxy / TARGET_RES_DEG) * TARGET_RES_DEG

width = int(round((right - left) / TARGET_RES_DEG))
height = int(round((top - bottom) / TARGET_RES_DEG))

transform = from_origin(
    left,
    top,
    TARGET_RES_DEG,
    TARGET_RES_DEG,
)

print("\nTarget grid:")
print("Width :", width)
print("Height:", height)
print("Bounds:", (left, bottom, right, top))
print("Grid spacing:", TARGET_RES_DEG, "degree")

inside_mask = geometry_mask(
    gdf.geometry,
    transform=transform,
    invert=True,
    out_shape=(height, width),
    all_touched=False,
)

print("Pixels inside Khulna:", int(inside_mask.sum()))


# ============================================================
# 9. PREPARE ALL 2022 SOURCES
# ============================================================

sources = {}

# Precipitation
for product, folder_name in PRECIP_FOLDERS.items():
    folder = RAW_DIR / "precipitation" / folder_name

    if not folder.exists():
        raise FileNotFoundError(f"Missing precipitation folder:\n{folder}")

    mm = monthly_map(folder)

    print(
        f"{product:14s} | folder={folder_name:24s} | "
        f"parsed monthly rasters={len(mm)}"
    )

    for month in range(1, 13):
        ym = (TEST_YEAR, month)

        if ym not in mm:
            available_2022 = sorted(k for k in mm if k[0] == TEST_YEAR)
            raise FileNotFoundError(
                f"Missing {product} raster for {TEST_YEAR}-{month:02d}\n"
                f"Folder: {folder}\n"
                f"Parsed {TEST_YEAR} entries: {available_2022}"
            )

        sources[(product, month)] = mm[ym]

# Dynamic land
for pred in LAND_DYNAMIC:
    folder = RAW_DIR / "predictors" / pred

    if not folder.exists():
        raise FileNotFoundError(f"Missing predictor folder:\n{folder}")

    mm = monthly_map(folder)

    print(
        f"{pred:14s} | folder={pred:24s} | "
        f"parsed monthly rasters={len(mm)}"
    )

    for month in range(1, 13):
        ym = (TEST_YEAR, month)

        if ym not in mm:
            available_2022 = sorted(k for k in mm if k[0] == TEST_YEAR)
            raise FileNotFoundError(
                f"Missing {pred} raster for {TEST_YEAR}-{month:02d}\n"
                f"Folder: {folder}\n"
                f"Parsed {TEST_YEAR} entries: {available_2022}"
            )

        sources[(pred, month)] = mm[ym]

# Static land
sources[("DEM", 0)] = DEM_PATH
sources[("Distance_Sea", 0)] = DFS_PATH


# ============================================================
# 10. SOURCE COUNT CHECK
# ============================================================

EXPECTED_SOURCES = 122

print("\n========================================")
print("SOURCE PREPARATION COMPLETE")
print("========================================")
print("Total sources prepared:", len(sources))
print("Expected sources:", EXPECTED_SOURCES)

if len(sources) != EXPECTED_SOURCES:
    raise ValueError(
        f"Expected {EXPECTED_SOURCES} sources, but found {len(sources)}."
    )

print("All 122 required sources found.")


# ============================================================
# 11. OUTPUT SETTINGS
# ============================================================

aligned_root = PROCESSED_DIR / "test2022_target_grid"
aligned_root.mkdir(parents=True, exist_ok=True)

profile = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 1,
    "dtype": "float32",
    "crs": WGS84_PROJ,
    "transform": transform,
    "nodata": DST_NODATA,
    "compress": "deflate",
    "predictor": 3,
}


# ============================================================
# 12. ALIGN ONE RASTER
# ============================================================

def align_one(src_path, out_path):
    out_path.parent.mkdir(parents=True, exist_ok=True)

    dst = np.full(
        (height, width),
        DST_NODATA,
        dtype=np.float32,
    )

    with rasterio.open(src_path) as src:
        src_crs = source_crs_or_verified_wgs84(src, src_path)

        src_arr = src.read(1, masked=True).astype(np.float32)

        TEMP_SRC_NODATA = -9999.0

        src_data = src_arr.filled(TEMP_SRC_NODATA).astype(np.float32)
        src_data[~np.isfinite(src_data)] = TEMP_SRC_NODATA

        reproject(
            source=src_data,
            destination=dst,
            src_transform=src.transform,
            src_crs=src_crs,
            src_nodata=TEMP_SRC_NODATA,
            dst_transform=transform,
            dst_crs=WGS84_PROJ,
            dst_nodata=DST_NODATA,
            resampling=Resampling.bilinear,
            init_dest_nodata=True,
        )

    # Keep coverage gaps as NoData and mask outside Khulna.
    dst[~inside_mask] = DST_NODATA

    valid = (
        inside_mask
        & np.isfinite(dst)
        & (~np.isclose(dst, DST_NODATA))
    )

    with rasterio.open(out_path, "w", **profile) as out:
        out.write(dst, 1)

    total_inside = int(inside_mask.sum())
    valid_inside = int(valid.sum())

    valid_inside_pct = (
        (valid_inside / total_inside) * 100.0
        if total_inside > 0 else np.nan
    )

    if valid_inside > 0:
        values = dst[valid]
        min_value = float(np.nanmin(values))
        max_value = float(np.nanmax(values))
        mean_value = float(np.nanmean(values))
    else:
        min_value = np.nan
        max_value = np.nan
        mean_value = np.nan

    return {
        "source": str(src_path),
        "output": str(out_path),
        "valid_inside_pixels": valid_inside,
        "total_inside_pixels": total_inside,
        "valid_inside_pct": valid_inside_pct,
        "min": min_value,
        "max": max_value,
        "mean": mean_value,
    }


# ============================================================
# 13. PROCESS ALL SOURCES
# ============================================================

qc_rows = []

for (name, month), src_path in sources.items():

    if month == 0:
        out_path = aligned_root / "static" / f"{name}.tif"
        label = f"{name} STATIC"
    else:
        out_path = (
            aligned_root
            / name
            / f"{name}_{TEST_YEAR}_{month:02d}.tif"
        )
        label = f"{name} {TEST_YEAR}-{month:02d}"

    print("Processing:", label)

    result = align_one(
        src_path,
        out_path,
    )

    qc_rows.append(result)


# ============================================================
# 14. QC TABLE
# ============================================================

qc = pd.DataFrame(qc_rows)

display(qc)

qc_path = aligned_root / "alignment_qc.csv"
qc.to_csv(qc_path, index=False)


# ============================================================
# 15. COVERAGE CHECK
# ============================================================

low = qc[qc["valid_inside_pct"] < 95].copy()

if len(low) > 0:
    print(
        "\nWARNING: Some rasters have less than "
        "95% valid coverage inside Khulna."
    )
    print(
        "Do NOT fill these gaps with nearest-neighbour values."
    )

    display(
        low[
            [
                "source",
                "valid_inside_pct",
                "min",
                "max",
                "mean",
            ]
        ]
    )
else:
    print("\nAll target-grid inputs have >=95% valid coverage.")


# ============================================================
# 16. FINAL SUMMARY
# ============================================================

print("\n==========================================")
print("2022 TARGET GRID ALIGNMENT COMPLETE")
print("==========================================")
print("Total processed rasters:", len(qc))
print("Expected rasters:", EXPECTED_SOURCES)
print("QC file:", qc_path)
print("Aligned rasters saved to:", aligned_root)

if len(qc) == EXPECTED_SOURCES:
    print(
        "\nSUCCESS: All 122 required 2022 input rasters were processed."
    )


PROJECT_ROOT = E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh
Filename year-month parser: OK

Canonical DEM:
E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\predictors\DEM\Khulna_SRTM_DEM.tif

Canonical Distance to Sea:
E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\predictors\Distance_Sea\Distance_Sea.tif

Boundary file:
E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\boundary\Khulna.shp

Target grid:
Width : 105
Height: 271
Bounds: (89.235, 21.66, 89.76, 23.015)
Grid spacing: 0.005 degree
Pixels inside Khulna: 15636
CCS            | folder=CCS                      | parsed monthly rasters=72
PDIR           | folder=PDIR                     |

,source,output,valid_inside_pixels,total_inside_pixels,valid_inside_pct,min,max,mean
0,E:\Geospatial\Precipitation-Downscaling-Khulna...,E:\Geospatial\Precipitation-Downscaling-Khulna...,14819,15636,94.774878,0.000000,26.871094,4.742375
1,E:\Geospatial\Precipitation-Downscaling-Khulna...,E:\Geospatial\Precipitation-Downscaling-Khulna...,14819,15636,94.774878,0.000000,19.937500,2.048472
2,E:\Geospatial\Precipitation-Downscaling-Khulna...,E:\Geospatial\Precipitation-Downscaling-Khulna...,14819,15636,94.774878,0.000000,10.222656,0.965554
3,E:\Geospatial\Precipitation-Downscaling-Khulna...,E:\Geospatial\Precipitation-Downscaling-Khulna...,14819,15636,94.774878,0.000000,12.523438,1.975626
4,E:\Geospatial\Precipitation-Downscaling-Khulna...,E:\Geospatial\Precipitation-Downscaling-Khulna...,14819,15636,94.774878,29.000000,145.164062,79.835892
...,...,...,...,...,...,...,...,...
117,E:\Geospatial\Precipitation-Downscaling-Khulna...,E:\Geospatial\Precipitation-Downscaling-Khulna...,14008,15636,89.588130,24.428797,32.294632,27.784445
118,E:\Geospatial\Precipitation-Downscaling-Khulna...,E:\Geospatial\Precipitation-Downscaling-Khulna...,14008,15636,89.588130,24.507010,30.291086,25.652864
119,E:\Geospatial\Precipitation-Downscaling-Khulna...,E:\Geospatial\Precipitation-Downscaling-Khulna...,14008,15636,89.588130,20.978458,25.605762,22.851723
120,E:\Geospatial\Precipitation-Downscaling-Khulna...,E:\Geospatial\Precipitation-Downscaling-Khulna...,15636,15636,100.000000,-0.000016,15.279215,4.859379



Do NOT fill these gaps with nearest-neighbour values.


,source,valid_inside_pct,min,max,mean
0,E:\Geospatial\Precipitation-Downscaling-Khulna...,94.774878,0.000000,26.871094,4.742375
1,E:\Geospatial\Precipitation-Downscaling-Khulna...,94.774878,0.000000,19.937500,2.048472
2,E:\Geospatial\Precipitation-Downscaling-Khulna...,94.774878,0.000000,10.222656,0.965554
3,E:\Geospatial\Precipitation-Downscaling-Khulna...,94.774878,0.000000,12.523438,1.975626
4,E:\Geospatial\Precipitation-Downscaling-Khulna...,94.774878,29.000000,145.164062,79.835892
...,...,...,...,...,...
116,E:\Geospatial\Precipitation-Downscaling-Khulna...,88.788693,21.949150,35.016968,28.167221
117,E:\Geospatial\Precipitation-Downscaling-Khulna...,89.588130,24.428797,32.294632,27.784445
118,E:\Geospatial\Precipitation-Downscaling-Khulna...,89.588130,24.507010,30.291086,25.652864
119,E:\Geospatial\Precipitation-Downscaling-Khulna...,89.588130,20.978458,25.605762,22.851723



2022 TARGET GRID ALIGNMENT COMPLETE
Total processed rasters: 122
Expected rasters: 122
QC file: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\test2022_target_grid\alignment_qc.csv
Aligned rasters saved to: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\test2022_target_grid

SUCCESS: All 122 required 2022 input rasters were processed.
